# AlphaFold Codec Fine-tuning Tutorial

This notebook demonstrates how to use the fine-tuning framework to adapt protein structure prediction models for various downstream tasks.

## Overview

We support 50+ fine-tuning tasks across 8 categories:
- **Drug Discovery**: Binding affinity, virtual screening, ADMET
- **Protein Engineering**: Stability, solubility, mutation effects
- **Antibody Design**: Affinity maturation, humanization, developability
- **Enzyme Engineering**: Activity, specificity, directed evolution
- **Protein-Protein Interactions**: Binding, interface, hot spots
- **Function Prediction**: GO terms, EC numbers, localization
- **Immunology**: B-cell epitopes, T-cell epitopes, immunogenicity
- **Structure Quality**: pLDDT, pAE, disorder prediction

In [ ]:
# Install dependencies if needed
# !pip install torch numpy

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np

# Core framework imports
from finetuning.configs import (
    FineTuningConfig,
    TrainingConfig,
    LoRAConfig,
    get_task_config,
    list_tasks_by_category,
    TASK_PRESETS,
)

# Check available task categories
print("Available task categories:")
for category, tasks in list_tasks_by_category().items():
    print(f"  {category}: {tasks}")

## 2. Understanding LoRA (Low-Rank Adaptation)

LoRA is a parameter-efficient fine-tuning technique that decomposes weight updates into low-rank matrices.

Instead of updating a full weight matrix W ∈ R^(d×k), we learn two smaller matrices:
- A ∈ R^(d×r)
- B ∈ R^(r×k)

Where r << min(d, k) is the rank.

The forward pass becomes: `y = Wx + (α/r) * BAx`

In [ ]:
# NumPy implementation to understand LoRA
class LoRALinearNumPy:
    """Educational NumPy implementation of LoRA."""
    
    def __init__(self, in_features: int, out_features: int, rank: int = 8, alpha: float = 16.0):
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # Original weights (frozen)
        self.W = np.random.randn(out_features, in_features) * 0.02
        
        # LoRA matrices (trainable)
        self.lora_A = np.random.randn(rank, in_features) * 0.02
        self.lora_B = np.zeros((out_features, rank))  # Initialize B to zero
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """Forward pass with LoRA."""
        # Original computation
        original = x @ self.W.T
        
        # LoRA contribution: x @ A.T @ B.T * scaling
        lora_contribution = x @ self.lora_A.T @ self.lora_B.T * self.scaling
        
        return original + lora_contribution
    
    def count_parameters(self):
        """Count trainable vs frozen parameters."""
        frozen = self.W.size
        trainable = self.lora_A.size + self.lora_B.size
        return frozen, trainable


# Example: Apply LoRA to a large attention layer
in_features = 384  # Typical hidden dimension
out_features = 384
rank = 8  # Low rank

lora_layer = LoRALinearNumPy(in_features, out_features, rank=rank)
frozen, trainable = lora_layer.count_parameters()

print(f"Original parameters: {frozen:,}")
print(f"LoRA parameters: {trainable:,}")
print(f"Parameter reduction: {trainable / frozen * 100:.2f}%")

# Test forward pass
x = np.random.randn(32, 128, in_features)  # [batch, seq_len, features]
y = lora_layer.forward(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {y.shape}")

## 3. Task-Specific Heads

Each task requires a specialized prediction head. Let's explore different head architectures.

### 3.1 Binding Affinity Head

Predicts protein-ligand binding strength (pKd, pIC50, ΔG).

In [ ]:
# NumPy implementation of Affinity Head
class AffinityHeadNumPy:
    """Educational NumPy implementation of binding affinity prediction head."""
    
    def __init__(self, pair_dim: int = 128, hidden_dim: int = 256):
        self.pair_dim = pair_dim
        self.hidden_dim = hidden_dim
        
        # Distance-based Gaussian smearing for spatial features
        self.gaussian_means = np.linspace(0, 20, 64)  # 0-20 Angstroms
        self.gaussian_std = 0.5
        
        # MLP for final prediction
        self.W1 = np.random.randn(hidden_dim, pair_dim + 64) * 0.02
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(1, hidden_dim) * 0.02
        self.b2 = np.zeros(1)
    
    def gaussian_smearing(self, distances: np.ndarray) -> np.ndarray:
        """Encode distances using Gaussian basis functions."""
        # distances: [N_protein, N_ligand]
        # Output: [N_protein, N_ligand, 64]
        distances_expanded = distances[..., np.newaxis]
        return np.exp(-((distances_expanded - self.gaussian_means) ** 2) / (2 * self.gaussian_std ** 2))
    
    def attention_pooling(self, features: np.ndarray, mask: np.ndarray = None) -> np.ndarray:
        """Pool over spatial dimensions using attention."""
        # Simple mean pooling for NumPy implementation
        if mask is not None:
            features = features * mask[..., np.newaxis]
            return features.sum(axis=(0, 1)) / mask.sum()
        return features.mean(axis=(0, 1))
    
    def forward(self, pair_repr: np.ndarray, distances: np.ndarray) -> dict:
        """Predict binding affinity.
        
        Args:
            pair_repr: Pair representation from model [N_protein, N_ligand, pair_dim]
            distances: Distance matrix [N_protein, N_ligand]
        """
        # Encode distances
        distance_features = self.gaussian_smearing(distances)
        
        # Concatenate with pair representation
        combined = np.concatenate([pair_repr, distance_features], axis=-1)
        
        # Pool to single vector
        pooled = self.attention_pooling(combined)
        
        # MLP prediction
        h = np.maximum(0, pooled @ self.W1.T + self.b1)  # ReLU
        affinity = h @ self.W2.T + self.b2
        
        return {
            'affinity_pred_value': float(affinity[0]),
            'distance_features': distance_features,
        }


# Example usage
head = AffinityHeadNumPy(pair_dim=128)

# Simulate input
n_protein, n_ligand = 100, 30
pair_repr = np.random.randn(n_protein, n_ligand, 128)
distances = np.random.uniform(2, 15, (n_protein, n_ligand))  # Angstroms

result = head.forward(pair_repr, distances)
print(f"Predicted pIC50: {result['affinity_pred_value']:.3f}")
print(f"Distance features shape: {result['distance_features'].shape}")

### 3.2 Protein Stability Head

Predicts the effect of mutations on protein stability (ΔΔG).

In [ ]:
class StabilityHeadNumPy:
    """Predict mutation effect on protein stability."""
    
    def __init__(self, single_dim: int = 384, hidden_dim: int = 256):
        # Project single representation
        self.W_wt = np.random.randn(hidden_dim, single_dim) * 0.02  # Wild-type
        self.W_mut = np.random.randn(hidden_dim, single_dim) * 0.02  # Mutant
        
        # Predict ΔΔG
        self.W_out = np.random.randn(1, hidden_dim * 2) * 0.02
        self.b_out = np.zeros(1)
    
    def forward(self, wt_repr: np.ndarray, mut_repr: np.ndarray, mutation_pos: int) -> dict:
        """Predict ΔΔG for a mutation.
        
        Args:
            wt_repr: Wild-type single representation [seq_len, single_dim]
            mut_repr: Mutant single representation [seq_len, single_dim]
            mutation_pos: Position of mutation
        """
        # Get representations at mutation site
        wt_site = wt_repr[mutation_pos]
        mut_site = mut_repr[mutation_pos]
        
        # Project through separate networks
        wt_h = np.maximum(0, wt_site @ self.W_wt.T)
        mut_h = np.maximum(0, mut_site @ self.W_mut.T)
        
        # Concatenate and predict
        combined = np.concatenate([wt_h, mut_h])
        ddg = combined @ self.W_out.T + self.b_out
        
        return {
            'ddg': float(ddg[0]),
            'is_destabilizing': float(ddg[0]) > 0,
        }


# Example
head = StabilityHeadNumPy(single_dim=384)

seq_len = 150
wt_repr = np.random.randn(seq_len, 384)
mut_repr = np.random.randn(seq_len, 384)  # Different due to mutation
mutation_pos = 50

result = head.forward(wt_repr, mut_repr, mutation_pos)
print(f"Predicted ΔΔG: {result['ddg']:.2f} kcal/mol")
print(f"Destabilizing: {result['is_destabilizing']}")

### 3.3 Enzyme Activity Head

Predicts kinetic parameters (kcat, Km, kcat/Km).

In [ ]:
class EnzymeActivityHeadNumPy:
    """Predict enzyme kinetic parameters."""
    
    def __init__(self, single_dim: int = 384, hidden_dim: int = 256, active_site_radius: float = 8.0):
        self.active_site_radius = active_site_radius
        
        # Active site encoder
        self.W_active = np.random.randn(hidden_dim, single_dim) * 0.02
        
        # Substrate encoder (simplified)
        self.W_substrate = np.random.randn(hidden_dim, 128) * 0.02
        
        # Output: kcat, Km, kcat/Km (in log scale)
        self.W_out = np.random.randn(3, hidden_dim * 2) * 0.02
        self.b_out = np.zeros(3)
    
    def get_active_site_residues(self, coords: np.ndarray, catalytic_site: int) -> np.ndarray:
        """Get residue indices within active site radius."""
        distances = np.linalg.norm(coords - coords[catalytic_site], axis=-1)
        return np.where(distances < self.active_site_radius)[0]
    
    def forward(self, single_repr: np.ndarray, coords: np.ndarray, 
                catalytic_site: int, substrate_features: np.ndarray) -> dict:
        """Predict kinetic parameters.
        
        Args:
            single_repr: Single representation [seq_len, single_dim]
            coords: CA coordinates [seq_len, 3]
            catalytic_site: Known catalytic residue index
            substrate_features: Substrate embedding [128]
        """
        # Get active site residues
        active_site_idx = self.get_active_site_residues(coords, catalytic_site)
        active_site_repr = single_repr[active_site_idx].mean(axis=0)
        
        # Encode
        active_h = np.maximum(0, active_site_repr @ self.W_active.T)
        substrate_h = np.maximum(0, substrate_features @ self.W_substrate.T)
        
        # Predict
        combined = np.concatenate([active_h, substrate_h])
        kinetics_log = combined @ self.W_out.T + self.b_out
        
        return {
            'log_kcat': float(kinetics_log[0]),
            'log_Km': float(kinetics_log[1]),
            'log_kcat_over_Km': float(kinetics_log[2]),
            'kcat': 10 ** float(kinetics_log[0]),
            'Km_uM': 10 ** float(kinetics_log[1]),
            'active_site_size': len(active_site_idx),
        }


# Example
head = EnzymeActivityHeadNumPy()

seq_len = 200
single_repr = np.random.randn(seq_len, 384)
coords = np.random.randn(seq_len, 3) * 10  # Random coordinates
catalytic_site = 100
substrate_features = np.random.randn(128)

result = head.forward(single_repr, coords, catalytic_site, substrate_features)
print(f"Predicted kcat: {result['kcat']:.1f} s⁻¹")
print(f"Predicted Km: {result['Km_uM']:.1f} μM")
print(f"Active site residues: {result['active_site_size']}")

### 3.4 B-cell Epitope Head

Predicts antibody binding epitopes on protein surfaces.

In [ ]:
class BcellEpitopeHeadNumPy:
    """Predict B-cell epitope residues."""
    
    def __init__(self, single_dim: int = 384, hidden_dim: int = 128):
        # Per-residue classifier
        self.W1 = np.random.randn(hidden_dim, single_dim) * 0.02
        self.W2 = np.random.randn(1, hidden_dim) * 0.02
    
    def forward(self, single_repr: np.ndarray, sasa: np.ndarray = None) -> dict:
        """Predict per-residue epitope probability.
        
        Args:
            single_repr: Single representation [seq_len, single_dim]
            sasa: Solvent accessible surface area [seq_len] (optional)
        """
        # Per-residue prediction
        h = np.maximum(0, single_repr @ self.W1.T)
        logits = (h @ self.W2.T).squeeze(-1)
        
        # Sigmoid for probability
        probs = 1 / (1 + np.exp(-logits))
        
        # Filter by surface accessibility if provided
        if sasa is not None:
            surface_mask = sasa > 25.0  # Typical SASA threshold
            probs = probs * surface_mask
        
        # Get top epitope residues
        top_indices = np.argsort(probs)[-20:][::-1]  # Top 20
        
        return {
            'epitope_probs': probs,
            'top_epitope_residues': top_indices,
            'top_probs': probs[top_indices],
        }


# Example
head = BcellEpitopeHeadNumPy()

seq_len = 300
single_repr = np.random.randn(seq_len, 384)
sasa = np.random.uniform(0, 100, seq_len)  # Random SASA values

result = head.forward(single_repr, sasa)
print(f"Top 10 epitope residues: {result['top_epitope_residues'][:10]}")
print(f"Their probabilities: {result['top_probs'][:10]}")

## 4. Training Loop

Here's how the training loop works with LoRA fine-tuning.

In [ ]:
class SimpleTrainer:
    """Simplified NumPy trainer for demonstration."""
    
    def __init__(self, learning_rate: float = 1e-4):
        self.learning_rate = learning_rate
        self.losses = []
    
    def compute_loss(self, predictions: np.ndarray, targets: np.ndarray) -> float:
        """MSE loss for regression tasks."""
        return float(np.mean((predictions - targets) ** 2))
    
    def compute_gradients(self, predictions: np.ndarray, targets: np.ndarray) -> np.ndarray:
        """Gradient of MSE loss."""
        return 2 * (predictions - targets) / len(predictions)
    
    def train_step(self, model, batch_x, batch_y):
        """Single training step."""
        # Forward pass
        predictions = model.forward(batch_x)
        
        # Compute loss
        loss = self.compute_loss(predictions, batch_y)
        self.losses.append(loss)
        
        # In real implementation, would compute gradients and update LoRA params
        return loss
    
    def train_epoch(self, model, data_x, data_y, batch_size: int = 32):
        """Train for one epoch."""
        n_samples = len(data_x)
        indices = np.random.permutation(n_samples)
        
        epoch_loss = 0
        n_batches = 0
        
        for i in range(0, n_samples, batch_size):
            batch_idx = indices[i:i + batch_size]
            loss = self.train_step(model, data_x[batch_idx], data_y[batch_idx])
            epoch_loss += loss
            n_batches += 1
        
        return epoch_loss / n_batches


# Demonstrate training loop
print("Training loop demonstration:")
print("-" * 40)

# Simulated data
n_samples = 1000
feature_dim = 384
X = np.random.randn(n_samples, feature_dim)
y = np.random.randn(n_samples)  # Affinity values

# Create a simple linear model with LoRA
class SimpleAffinityModel:
    def __init__(self, dim):
        self.W = np.random.randn(1, dim) * 0.02
        self.lora = LoRALinearNumPy(dim, 1, rank=4)
    
    def forward(self, x):
        return self.lora.forward(x).squeeze(-1)

model = SimpleAffinityModel(feature_dim)
trainer = SimpleTrainer(learning_rate=1e-4)

# Train for a few epochs
for epoch in range(5):
    loss = trainer.train_epoch(model, X, y)
    print(f"Epoch {epoch + 1}: Loss = {loss:.4f}")

## 5. Using the Full Framework (PyTorch)

Here's how to use the complete fine-tuning framework with PyTorch.

In [ ]:
# Full PyTorch example (requires PyTorch installed)
pytorch_example = '''
import torch
from finetuning import FineTuningConfig, TrainingConfig, Trainer
from finetuning.modules import LoRAModule
from finetuning.heads import AffinityHead, AffinityHeadConfig
from finetuning.data import AffinityDataset
from finetuning.configs import get_task_config

# 1. Load your base model (Boltz, AF2, etc.)
base_model = load_pretrained_model("boltz-1")

# 2. Configure fine-tuning
config = FineTuningConfig(
    strategy="lora",
    task="binding_affinity",
    lora_rank=8,
    training=TrainingConfig(
        learning_rate=5e-5,
        max_steps=10000,
        batch_size=8,
        gradient_accumulation_steps=4,
        fp16=True,
    ),
)

# 3. Apply LoRA
lora_model = LoRAModule(
    base_model,
    rank=config.lora_rank,
    alpha=16.0,
    target_modules=["q_proj", "k_proj", "v_proj"],
)

# 4. Add task-specific head
task_config = get_task_config("binding_affinity")
head = AffinityHead(AffinityHeadConfig(
    pair_dim=base_model.config.pair_dim,
    hidden_dim=256,
    use_gaussian_smearing=True,
    use_attention_pooling=True,
))

# 5. Load data
train_dataset = AffinityDataset(
    data_dir="./pdbbind/train",
    affinity_file="affinities.csv",
)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=config.training.batch_size, shuffle=True
)

# 6. Create trainer
trainer = Trainer(
    model=lora_model,
    head=head,
    config=config,
    train_loader=train_loader,
)

# 7. Train
trainer.train()

# 8. Evaluate
metrics = trainer.evaluate(test_loader)
print(f"Test RMSE: {metrics[\"rmse\"]:.3f}")
print(f"Test Pearson: {metrics[\"pearson\"]:.3f}")

# 9. Save
trainer.save_checkpoint("binding_affinity_lora.pt")
'''

print("PyTorch Fine-tuning Example:")
print("=" * 50)
print(pytorch_example)

## 6. Task Configuration Reference

Quick reference for all supported tasks and their configurations.

In [ ]:
# Task configuration reference
task_reference = {
    "Drug Discovery": {
        "binding_affinity": {
            "outputs": ["pKd", "pIC50", "ΔG"],
            "metrics": ["RMSE", "Pearson", "R²"],
            "datasets": ["PDBbind", "BindingDB", "ChEMBL"],
        },
        "virtual_screening": {
            "outputs": ["hit_probability", "rank"],
            "metrics": ["AUROC", "Enrichment Factor", "BEDROC"],
            "datasets": ["DUD-E", "LIT-PCBA"],
        },
    },
    "Protein Engineering": {
        "stability": {
            "outputs": ["ΔΔG", "Tm_shift"],
            "metrics": ["RMSE", "Spearman"],
            "datasets": ["ProTherm", "FireProtDB", "Megascale"],
        },
        "mutation_effects": {
            "outputs": ["fitness", "pathogenicity"],
            "metrics": ["Spearman", "AUROC"],
            "datasets": ["DMS datasets", "ClinVar", "gnomAD"],
        },
    },
    "Antibody Design": {
        "affinity_maturation": {
            "outputs": ["ΔΔG", "fold_improvement"],
            "metrics": ["Spearman", "Top-k accuracy"],
            "datasets": ["SAbDab", "SKEMPI"],
        },
        "developability": {
            "outputs": ["aggregation", "viscosity", "expression"],
            "metrics": ["AUROC", "RMSE"],
            "datasets": ["TAP", "Internal assays"],
        },
    },
    "Enzyme Engineering": {
        "activity": {
            "outputs": ["kcat", "Km", "kcat/Km"],
            "metrics": ["RMSE", "Spearman"],
            "datasets": ["BRENDA", "SABIO-RK"],
        },
        "specificity": {
            "outputs": ["substrate_profile", "selectivity"],
            "metrics": ["Spearman", "AUROC"],
            "datasets": ["MEROPS", "CAZy"],
        },
    },
    "Immunology": {
        "bcell_epitope": {
            "outputs": ["epitope_prob_per_residue"],
            "metrics": ["AUROC", "AUPRC"],
            "datasets": ["IEDB", "BepiPred"],
        },
        "tcell_epitope": {
            "outputs": ["MHC_binding", "presentation"],
            "metrics": ["AUROC", "PPV"],
            "datasets": ["IEDB", "NetMHCpan"],
        },
    },
}

print("Task Reference Guide")
print("=" * 60)
for category, tasks in task_reference.items():
    print(f"\n{category}")
    print("-" * 40)
    for task_name, info in tasks.items():
        print(f"  {task_name}:")
        print(f"    Outputs: {info['outputs']}")
        print(f"    Metrics: {info['metrics']}")
        print(f"    Datasets: {info['datasets']}")

## 7. Best Practices

### Choosing LoRA Rank
- **rank=4**: Minimal capacity, best for small datasets (<500 samples)
- **rank=8**: Default, good balance for most tasks
- **rank=16-32**: Higher capacity, for complex tasks with >1000 samples

### Learning Rate
- **LoRA**: 1e-4 to 5e-4 (can use higher LR than full fine-tuning)
- **Full fine-tuning**: 1e-5 to 5e-5
- **Head-only**: 1e-3 to 5e-4

### Data Augmentation
- Rotate/translate structures randomly
- MSA subsampling for robustness
- Add noise to coordinates during training

### Evaluation
- Always use held-out test set with no sequence similarity to training
- Report confidence intervals
- Compare against baseline methods

## Summary

This tutorial covered:

1. **LoRA fundamentals**: How low-rank adaptation works and why it's efficient
2. **Task-specific heads**: Architectures for affinity, stability, enzymes, and epitopes
3. **Training workflow**: From configuration to evaluation
4. **Best practices**: Hyperparameter selection and evaluation

For more details, see:
- `finetuning/FINETUNING_GUIDE.md`: Complete documentation
- `finetuning/heads/`: All prediction head implementations
- `finetuning/modules/`: LoRA, Adapter, and Prompt Tuning implementations